
# Annual Growth Rate
- This calculates how much the price changes on averages and applies that to the future prices


In [195]:
print(data.columns)                                                                                                      

Index(['date', 'Region', 'County', 'market', 'latitude', 'longitude',
       'category', 'commodity', 'unit', 'price'],
      dtype='str')


In [196]:
data["date"] = pd.to_datetime(data["date"], errors="coerce")

In [197]:
# clean dataset
data = data.dropna(subset=["date"])
# this removes rows where data is invalid


In [198]:
# Extract Year
data["Year"] = data["date"].dt.year

In [199]:
print(data.columns)

Index(['date', 'Region', 'County', 'market', 'latitude', 'longitude',
       'category', 'commodity', 'unit', 'price', 'Year'],
      dtype='str')


In [200]:
# Calculate the average price per Year 
average_price_per_year = data.groupby(["Year","commodity"])["price"].mean().reset_index()

# Filter one commodity, predictictions are usually done per commodity
# Show the available commodities
commodity = average_price_per_year['commodity'].unique()


In [134]:
print("available commodity:",commodity)
# This displays the available commodities to the user

available commodity: <StringArray>
[                     'Beans',                'Beans (dry)',
                      'Bread',                      'Maize',
              'Maize (white)',    'Milk (cow, pasteurized)',
            'Oil (vegetable)',           'Potatoes (Irish)',
                    'Sorghum',              'Fuel (diesel)',
            'Fuel (kerosene)',     'Fuel (petrol-gasoline)',
                'Maize flour',                       'Rice',
                       'Salt',                    'Bananas',
                      'Sugar',                'Wheat flour',
                'Cooking fat',                 'Milk (UHT)',
               'Onions (red)',                   'Tomatoes',
           'Beans (dolichos)',             'Beans (kidney)',
               'Beans (mung)',           'Beans (rosecoco)',
             'Beans (yellow)',                    'Cabbage',
              'Cowpea leaves',                    'Cowpeas',
          'Fish (omena, dry)',                    

In [218]:
# The user inputs the commodity of choice
commodity = input("Enter commodity")

In [219]:
# Filter the data for the selected commodity
data = average_price_per_year[average_price_per_year["commodity"]==commodity]

In [ ]:
# Calculating the Growth Rate
data["GrowthRate"] = data["price"].pct_change()
# .pct_change()is the fractional change between the current price and the prior price
average_growth = data["GrowthRate"].mean()
# This calculates the average growth rate


In [229]:
# The user inputs the year they'd like to predict the price
Year = int(input("Enter the year you'd like to predict the price"))
# Get the dat of the last year before predicted year
Last_Year = data["Year"].iloc[-1]
# .iloc[-1] gets the data of the latest year in the year column
# Get the latest price in the price column
Latest_Price = data["price"].iloc[-1]
# Calculate years into the future(ahead)
Year_ahead = Year - Last_Year
# Get the future price
Future_Price = Latest_Price*(1+average_growth)**Year_ahead



In [230]:
print("Future_Price:",Future_Price)

Future_Price: 61.38784135368566


In [231]:
import pandas as pd
data = pd.read_csv("Cleaned_Data.csv")
print(data)



            date         Region   County        market  latitude  longitude  \
0      1/15/2006          Coast  MOMBASA       Mombasa     -4.05      39.67   
1      1/15/2006          Coast  MOMBASA       Mombasa     -4.05      39.67   
2      1/15/2006          Coast  MOMBASA       Mombasa     -4.05      39.67   
3      1/15/2006          Coast  MOMBASA       Mombasa     -4.05      39.67   
4      1/15/2006        Eastern    KITUI         Kitui     -1.37      38.02   
...          ...            ...      ...           ...       ...        ...   
18551  3/15/2026  North Eastern  GARISSA  IFO (Daadab)      0.11      40.32   
18552  3/15/2026  North Eastern  GARISSA  IFO (Daadab)      0.11      40.32   
18553  3/15/2026  North Eastern  GARISSA  IFO (Daadab)      0.11      40.32   
18554  3/15/2026  North Eastern  GARISSA  IFO (Daadab)      0.11      40.32   
18555  3/15/2026  North Eastern  GARISSA  IFO (Daadab)      0.11      40.32   

                    category        commodity   uni

## Predicting Demand

In [232]:
# first get the elasticity of the different categories 
def elasticity_by_category(category):
    category = category.lower() # puts your category in lowercase

    if category == "cereals and tubers":
        return -0.6
    elif category == "pulses and nuts":
        return -0.8
    elif category == "vegetables and fruits":
        return -0.9
    elif category == "oils and fats":
        return -0.5
    elif category == "meat,fish and eggs":
        return -1.3
    elif category == "dairy":
        return -0.9
    else:
        return -0.8 # this is a default value
    
# get the category data from the dataset

commodity_data = data[data["commodity"] == commodity]
category = commodity_data["category"].iloc[0]
current_elasticity = elasticity_by_category(category)
    


In [224]:
# Getting the Demnad Index(this is a measure used to track how the demand of a product changes over time compared to a fixed starting point)
price_change_percentage = ((Future_Price-Latest_Price)/Latest_Price)*100
print("Price_Change_percentage:",price_change_percentage)

Price_Change_percentage: -43.39302225263468


In [225]:
# Using 100 as my baseline to represent the current market equilibrium
Demand_Index = 100*(1+(current_elasticity*price_change_percentage))
print("Demand_Index:",Demand_Index)
if Demand_Index>= 100:
    print("The demand for the commodity is high,Increase the stock levels to make more sales")
elif 90<= Demand_Index< 100:
    print("The demand for the commodity is stable,Ensure the stock level is enough")
else:
    print("The demand is low,do not but more stock")

Demand_Index: 2703.5813351580805
The demand for the commodity is high,Increase the stock levels to make more sales


# Predicting Profit

In [228]:
# Predicting profit
Cost_price = float(input(f"Enter current price for {commodity}:"))
Predicted_Profit = Future_Price - Cost_price
print("Predicted_Profit:",Predicted_Profit)
Percentage_profit = (Predicted_Profit/Cost_price)*100
print("Percentage_profit:",Percentage_profit)
if Percentage_profit>= 30:
    print("the profit gain is high")
elif Percentage_profit>= 20:
    print("The profit gain is good")
else:
    print("low profit gain")


Predicted_Profit: 11.387841353685658
Percentage_profit: 22.775682707371317
The profit gain is good


# Restock Alert System 

In [227]:
# The Restock alert system
# The user inputs the current stock and restock stock
Current_stock = int(input("Enter the current stock"))
Restock_stock = int(input("Enter when the stock level should be restocked"))
# when to restock
if Current_stock < Restock_stock:
    print("Time to Restock!")
    if Percentage_profit >= 20 and Demand_Index>=90:
        print("Restock needed")
    else:
        print("Restock not needed")
else:
    print("Do not Restock!")

Time to Restock!
Restock needed
